# Google News AI Scraper V3 (Metadata Only)

Simple scraper to extract Title, Link, Date, and Source Name within a date range.

In [15]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import json
import time

def parse_rss_date(date_str):
    """Parses RSS pubDate format: Sun, 04 Jan 2026 20:16:00 GMT"""
    if not date_str: return None
    try:
        # Standard RSS format
        dt = datetime.strptime(date_str, "%a, %d %b %Y %H:%M:%S %Z")
        return dt
    except ValueError:
        pass
    try:
        # Sometimes timezone is missing or different
        dt = datetime.strptime(date_str.rsplit(' ', 1)[0], "%a, %d %b %Y %H:%M:%S")
        return dt
    except ValueError:
        return None

def get_google_news_ai(start_date_str, end_date_str, limit=None):
    # RSS Feed URL for 'AI' (US English)
    rss_url = "https://news.google.com/rss/search?q=AI&hl=en-US&gl=US&ceid=US:en"
    
    # Parse Input Dates (YYYY-MM-DD)
    try:
        processed_start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
        processed_end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
        # Set end date to end of day
        processed_end_date = processed_end_date.replace(hour=23, minute=59, second=59)
    except ValueError:
        print("Invalid date format. Please use YYYY-MM-DD.")
        return json.dumps([], indent=4)

    print(f"Fetching Google News (AI)... Date Range: {start_date_str} to {end_date_str}")
    if limit:
        print(f"Limit set to: {limit} articles")

    try:
        response = requests.get(rss_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        if response.status_code != 200:
            print(f"Failed to fetch RSS feed. Status: {response.status_code}")
            return json.dumps([], indent=4, ensure_ascii=False)

        # Parse XML
        soup = BeautifulSoup(response.content, 'xml')
        items = soup.find_all('item')
        
        filtered_articles = []
        
        for item in items:
            # Check limit BEFORE processing if we want strict limit on *output*
            if limit and len(filtered_articles) >= limit:
                break

            # 1. Extract Fields
            title = item.title.get_text(strip=True)
            link = item.link.get_text(strip=True)
            pub_date_str = item.pubDate.get_text(strip=True)
            
            # Extract Source Name from <source> tag
            # <source url="...">Source Name</source>
            source_tag = item.source
            source_name = source_tag.get_text(strip=True) if source_tag else "Unknown"
            # source_url = source_tag.get('url') if source_tag else ""

            # 2. Parse Date
            article_date = parse_rss_date(pub_date_str)
            if not article_date:
                continue
            
            # 3. Filter by Date Range
            # Make article_date naive (remove timezone) for comparison
            article_date_naive = article_date.replace(tzinfo=None)
            
            if processed_start_date <= article_date_naive <= processed_end_date:
                article_data = {
                    "title": title,
                    "url": link,
                    "date": article_date.strftime("%Y-%m-%d %H:%M:%S"),
                    "source_name": source_name
                }
                filtered_articles.append(article_data)
        
        print(f"Found {len(filtered_articles)} articles in range.")
        
        # Return JSON
        return json.dumps(filtered_articles, indent=4, ensure_ascii=False)

    except Exception as e:
        print(f"Error: {e}")
        return json.dumps([], indent=4)

# Example Usage (Step 1)
if __name__ == "__main__":
    # Test with a date range covering 'today' or recent past
    step1_result = get_google_news_ai("2026-01-04", "2026-01-05", limit=3)
    print("\n[Step 1 Result] Metadata extracted:")
    print(step1_result)


Fetching Google News (AI)... Date Range: 2026-01-04 to 2026-01-05
Limit set to: 3 articles
Found 3 articles in range.

[Step 1 Result] Metadata extracted:
[
    {
        "title": "World ‘may not have time’ to prepare for AI safety risks, says leading researcher - The Guardian",
        "url": "https://news.google.com/rss/articles/CBMiyAFBVV95cUxNVUVLUVlTdTVvaVlKYTZJc0VuNVk1cW5HakhwX3BTVUZOMkpKTGxCSUVXcUFzbXhUVzNnNXg4MDFJQWtBdTByc2w5aktfNi05V2V3dUhFT3poWV9SaFh5ZjJSRHhmNW1rdWw0TnJYakZKN2NXLU9SY3Fnc2JneG8yVm14c1JveTZOd0FaQzhTUGJ1TUlFMlN3YkcxSDZ6U0JISDJHb0hEZHVhcWRnNGhELUg1aDRreU5YZlJNVU0yX0JWdnp4RHpKdg?oc=5",
        "date": "2026-01-04 20:16:00",
        "source_name": "The Guardian"
    },
    {
        "title": "Is the AI boom a bubble waiting to pop? Here’s what history says - Yahoo Finance",
        "url": "https://news.google.com/rss/articles/CBMifEFVX3lxTE9rdC1vc2xKV2RMMzRxU3NoNVA2SjJWSXE2TGowUS00aXQwSUFmSVUxVzZZZ1dOTzROZTRwTXJfRDZfNVJPR2trSmUyMkUtbnpZaW1qZXZPNnFHcjV6Q0NHd2ZnSURYd

In [14]:
# =========================================================
# STEP 2: Content Extraction (Debug Mode + Enhanced Headers)
# =========================================================
import re
import base64

def is_code_like_v3(text):
    """Returns True if text looks like JS/CSS code."""
    if "function()" in text or "function ()" in text: return True
    if "var " in text and "=" in text: return True
    if "{" in text and "}" in text and ";" in text: return True
    if "@media" in text or "@font-face" in text: return True
    symbols = text.count('{') + text.count('}') + text.count(';') + text.count('(') + text.count(')')
    if len(text) > 0 and (symbols / len(text)) > 0.05:
        return True
    return False

def decode_google_news_url(url, verbose=False):
    """Attempts to decode the Google News base64 ID to find the real URL."""
    try:
        match = re.search(r'articles/([a-zA-Z0-9_\-]+)', url)
        if not match:
            if verbose: print("   [Base64] No ID match in URL.")
            return None
        
        code = match.group(1)
        padding = len(code) % 4
        if padding > 0:
            code += '=' * (4 - padding)
            
        decoded_bytes = base64.urlsafe_b64decode(code)
        decoded_str = decoded_bytes.decode('latin1', errors='ignore') 
        
        urls = re.findall(r'(https?://[a-zA-Z0-9_\-\./]+)', decoded_str)
        
        best_link = None
        for link in urls:
            if "google.com" not in link and "googleusercontent" not in link:
                 if best_link is None or len(link) > len(best_link):
                     best_link = link
        return best_link
    except Exception as e:
        if verbose: print(f"   [Base64] Error: {e}")
        return None

def extract_article_content_v3(url, verbose=True):
    target_url = url
    
    # --- RESOLUTION STRATEGY 1: BASE64 DECODING ---
    if "news.google.com" in url:
        decoded_url = decode_google_news_url(url, verbose)
        if decoded_url:
            if verbose: print(f"   -> [Decoded] Target: {decoded_url[:60]}...")
            target_url = decoded_url
        else:
            if verbose: print("   -> [Decoded] Failed to decode base64.")
    
    session = requests.Session()
    # ENHANCED HEADERS to bypass basic bot detection
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Cache-Control": "max-age=0",
    }
    
    try:
        # --- REQUEST ---
        response = session.get(target_url, headers=headers, timeout=20, allow_redirects=True)
        
        if verbose: 
            print(f"   -> [Request] Status: {response.status_code}, Final URL: {response.url[:60]}...")
            
        html = response.text
        current_url = response.url

        # --- RESOLUTION STRATEGY 2: HTML FALLBACK (If Base64 failed or redirected back to Google) ---
        if "news.google.com" in current_url or "consent.google.com" in current_url:
            if verbose: print("   -> [Fallback] Stuck on Google. Searching HTML for link...")
            candidates = re.findall(r'(https?://[^"\'\s<>]+)', html)
            best_link = None
            for link in candidates:
                if "google.com" not in link and "gstatic" not in link and "google-analytics" not in link and "angular.dev" not in link:
                    if best_link is None or len(link) > len(best_link):
                        best_link = link
            
            if best_link:
                if verbose: print(f"   -> [Fallback] Found: {best_link[:60]}...")
                try:
                    response = session.get(best_link, headers=headers, timeout=20, allow_redirects=True)
                    html = response.text
                    if verbose: print(f"   -> [Fallback Request] Status: {response.status_code}")
                except Exception as e:
                    if verbose: print(f"   -> [Fallback Error] {e}")
                    return ""
            else:
                if verbose: print("   -> [Fallback] No link found in HTML.")

        # --- CONTENT EXTRACTION ---
        pattern = r'<p[^>]*>(.*?)</p>'
        matches = re.findall(pattern, html, re.DOTALL | re.IGNORECASE)
        
        if verbose: print(f"   -> [Extract] Found {len(matches)} <p> tags.\n")
        
        valid_paragraphs = []
        for m in matches:
            text = re.sub(r'<[^>]+>', '', m).strip()
            if len(text) > 40 and not is_code_like_v3(text) and "cookie" not in text.lower():
                valid_paragraphs.append(text)
        
        count = min(len(valid_paragraphs), 5)
        return "\n\n".join(valid_paragraphs[:count])
    except Exception as e:
        if verbose: print(f"   -> [Error] {e}")
        return ""

def enrich_articles_with_content(json_input):
    if isinstance(json_input, str):
        articles = json.loads(json_input)
    else:
        articles = json_input
    
    print(f"\n--- Processing Content for {len(articles)} articles ---")
    
    for i, article in enumerate(articles):
        url = article.get('url')
        title = article.get('title', 'No Title')
        
        print(f"[{i+1}/{len(articles)}] Extracting: {title[:30]}...")
        if url:
            # Force verbose=True based on user debugging need
            content = extract_article_content_v3(url, verbose=True)
            article['raw_content'] = content
        else:
            article['raw_content'] = ""
        time.sleep(1)
        
    return json.dumps(articles, indent=4, ensure_ascii=False)

# Example Usage (Step 2)
if __name__ == "__main__":
    # Check specific URL from user request
    test_url = 'https://news.google.com/rss/articles/CBMiyAFBVV95cUxNVUVLUVlTdTVvaVlKYTZJc0VuNVk1cW5HakhwX3BTVUZOMkpKTGxCSUVXcUFzbXhUVzNnNXg4MDFJQWtBdTByc2w5aktfNi05V2V3dUhFT3poWV9SaFh5ZjJSRHhmNW1rdWw0TnJYakZKN2NXLU9SY3Fnc2JneG8yVm14c1JveTZOd0FaQzhTUGJ1TUlFMlN3YkcxSDZ6U0JISDJHb0hEZHVhcWRnNGhELUg1aDRreU5YZlJNVU0yX0JWdnp4RHpKdg?oc=5'
    print("\n--- Testing User Specific URL (Standard + Base64 + Debug) ---")
    print(extract_article_content_v3(test_url)[:500])
    
    # Full Run
    if 'step1_result' in locals():
        final_json = enrich_articles_with_content(step1_result)
        print("\n[Final Result]")
        print(final_json)



--- Testing User Specific URL (Standard + Base64 + Debug) ---
   -> [Decoded] Failed to decode base64.
   -> [Request] Status: 200, Final URL: https://news.google.com/rss/articles/CBMiyAFBVV95cUxNVUVLUVl...
   -> [Fallback] Stuck on Google. Searching HTML for link...
   -> [Fallback] Found: https://lh3.googleusercontent.com/J6_coFbogxhRI9iM864NL_liGX...
   -> [Fallback Request] Status: 200
   -> [Extract] Found 0 <p> tags.



--- Processing Content for 3 articles ---
[1/3] Extracting: World ‘may not have time’ to p...
   -> [Decoded] Failed to decode base64.
   -> [Request] Status: 200, Final URL: https://news.google.com/rss/articles/CBMiyAFBVV95cUxNVUVLUVl...
   -> [Fallback] Stuck on Google. Searching HTML for link...
   -> [Fallback] Found: https://lh3.googleusercontent.com/J6_coFbogxhRI9iM864NL_liGX...
   -> [Fallback Request] Status: 200
   -> [Extract] Found 0 <p> tags.

[2/3] Extracting: Is the AI Boom a Bubble Waitin...
   -> [Decoded] Failed to decode base64.
   -> [Request] 